# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Soham334/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.



### Rule

I will prioritize content for review when it has not been updated recently and still has meaningful search visibility. The score is deliberately simple: stale content gets the stale flag, and visible content contributes its observed impressions. This is a decision-support baseline, not a claim that every selected page needs a refresh.
### Signal checks

I will first check two signals that the baseline rule relies on:

1. `days_since_last_update` as the staleness signal, which is linked to FlyRank's refresh/staleness flags.
2. `impressions_90d` as the visibility/volume signal, linked to the quick-win logic.

The signal tables show bucket sizes (`n`) and observed performance. The verdicts are based on the measured differences between buckets.

### Reason code

- `stale_visible` — the content is sufficiently old and still has meaningful search visibility.

### Action

- `REFRESH_REVIEW` — prioritize the item for human review of whether the content should be refreshed.

The rule does not use `trend_direction`, `trend_pct`, or any label-derived field.

In [2]:
!git clone https://github.com/Soham334/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 137 (delta 47), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 1.87 MiB | 6.04 MiB/s, done.
Resolving deltas: 100% (47/47), done.


In [3]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship


In [4]:
from pathlib import Path

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

print("Dataset exists:", DATA_PATH.exists())
print("Current folder:", Path.cwd())

Dataset exists: True
Current folder: /content/flyrank-ml-internship


In [10]:
# ---------------------------------------------------------
# SIGNAL CHECK 1 — STALENESS
# ---------------------------------------------------------

staleness_bins = [-np.inf, 90, 180, 365, np.inf]
staleness_labels = [
    "0-90 days",
    "91-180 days",
    "181-365 days",
    "366+ days"
]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=staleness_bins,
    labels=staleness_labels
)

staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions_90d=("impressions_90d", "median"),
          median_impressions_last_30d=("impressions_last_30d", "median"),
          median_ctr=("ctr", "median")
      )
      .reset_index()
)

print("SIGNAL CHECK 1 — STALENESS")
display(staleness_check)

recent_visibility = df.loc[
    df["days_since_last_update"] <= 90,
    "impressions_90d"
].median()

stale_visibility = df.loc[
    df["days_since_last_update"] >= 181,
    "impressions_90d"
].median()

if pd.isna(recent_visibility) or pd.isna(stale_visibility):
    staleness_verdict = "MIXED"
elif stale_visibility >= recent_visibility:
    staleness_verdict = "CONFIRMED"
else:
    staleness_verdict = "OPPOSITE"

print("Staleness verdict:", staleness_verdict)


# ---------------------------------------------------------
# SIGNAL CHECK 2 — SEARCH VOLUME / VISIBILITY
# ---------------------------------------------------------

volume_bins = [-np.inf, 0, 500, 5000, 25000, np.inf]
volume_labels = [
    "0",
    "1-500",
    "501-5000",
    "5001-25000",
    "25000+"
]

df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=volume_bins,
    labels=volume_labels
)

volume_check = (
    df.groupby("volume_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_clicks_90d=("clicks_90d", "median"),
          median_ctr=("ctr", "median"),
          median_impressions_last_30d=("impressions_last_30d", "median")
      )
      .reset_index()
)

print("\nSIGNAL CHECK 2 — SEARCH VISIBILITY / VOLUME")
display(volume_check)

# Compare items with meaningful visibility against items with
# little/no visibility.
high_volume_clicks = df.loc[
    df["impressions_90d"] >= 500,
    "clicks_90d"
].median()

low_volume_clicks = df.loc[
    df["impressions_90d"] < 500,
    "clicks_90d"
].median()

if pd.isna(high_volume_clicks) or pd.isna(low_volume_clicks):
    volume_verdict = "MIXED"
elif high_volume_clicks > low_volume_clicks:
    volume_verdict = "CONFIRMED"
elif high_volume_clicks < low_volume_clicks:
    volume_verdict = "OPPOSITE"
else:
    volume_verdict = "MIXED"

print("Volume/visibility verdict:", volume_verdict)


# ---------------------------------------------------------
# LEAKAGE CHECK
# ---------------------------------------------------------

forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("\nLeakage check:")
for feature in forbidden_features:
    print(f"PASS — {feature} is not used by the baseline rule.")

print("\nSignal checks complete.")

SIGNAL CHECK 1 — STALENESS


,staleness_bucket,n,median_impressions_90d,median_impressions_last_30d,median_ctr
0,0-90 days,20655,472.0,86.0,0.04
1,91-180 days,9171,1692.0,305.0,0.10
2,181-365 days,169,16.0,4.0,0.00
3,366+ days,5,2.0,0.0,0.00


Staleness verdict: OPPOSITE

SIGNAL CHECK 2 — SEARCH VISIBILITY / VOLUME


,volume_bucket,n,median_clicks_90d,median_ctr,median_impressions_last_30d
0,0,0,NaN,NaN,NaN
1,1-500,13285,0.0,0.00,6.0
2,501-5000,10565,2.0,0.14,311.0
3,5001-25000,4773,21.0,0.22,2275.0
4,25000+,1377,103.0,0.23,11927.0


Volume/visibility verdict: CONFIRMED

Leakage check:
PASS — trend_direction is not used by the baseline rule.
PASS — trend_pct is not used by the baseline rule.
PASS — is_declining_label is not used by the baseline rule.

Signal checks complete.


## 2. Build the ranked queue (writes the CSV)

### Baseline score

The baseline uses one transparent rule:

`stale × visible impressions`

An item is considered stale when `days_since_last_update >= 180`. Visibility is measured using `impressions_90d`. The score is therefore easy to reproduce and explain: stale items with more observed visibility rank higher.

The action is `REFRESH_REVIEW` and the reason code is `stale_visible`.

The queue contains all content items, ranked by the baseline score. Items that do not meet the rule receive a score of zero and are not assigned the action.

In [7]:
# ---------------------------------------------------------
# BUILD ONE TRANSPARENT BASELINE RULE
# ---------------------------------------------------------

# Work from the original dataset columns.
baseline = df.copy()

# Simple, human-readable conditions.
baseline["stale"] = (
    baseline["days_since_last_update"] >= 180
).astype(int)

baseline["visible"] = (
    baseline["impressions_90d"] > 0
).astype(int)

# Transparent score: no fitted weights.
baseline["score"] = (
    baseline["stale"]
    * baseline["visible"]
    * baseline["impressions_90d"]
)

# One reason code for scored items.
baseline["reason_code"] = np.where(
    baseline["score"] > 0,
    "stale_visible",
    ""
)

# One action label.
baseline["action"] = np.where(
    baseline["score"] > 0,
    "REFRESH_REVIEW",
    ""
)

# Rank highest score first.
baseline = baseline.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

baseline["rank"] = np.arange(1, len(baseline) + 1)

# Select the queue columns.
queue = baseline[
    [
        "rank",
        "content_id",
        "client_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d",
        "impressions_last_30d",
        "ctr",
        "avg_position"
    ]
].copy()

# Create output directory.
OUTPUT_PATH = Path("work/outputs")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

CSV_PATH = OUTPUT_PATH / "baseline_action_score.csv"

# Write the ranked queue.
queue.to_csv(CSV_PATH, index=False)

print("Queue written to:", CSV_PATH)
print("Queue rows:", len(queue))
print("Scored rows:", (queue["score"] > 0).sum())

print("\nTop 10:")
display(queue.head(10))

Queue written to: work/outputs/baseline_action_score.csv
Queue rows: 30000
Scored rows: 174

Top 10:


,rank,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d,impressions_last_30d,ctr,avg_position
0,1,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_visible,REFRESH_REVIEW,194,61678,3864,0.15,19.7
1,2,content_7368877ea310,client_7f2253d7e2,59472,stale_visible,REFRESH_REVIEW,194,59472,3778,0.13,24.8
2,3,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_visible,REFRESH_REVIEW,194,25715,2305,0.23,22.2
3,4,content_0a91db491d14,client_7f2253d7e2,13299,stale_visible,REFRESH_REVIEW,193,13299,2246,0.49,10.5
4,5,content_5feee3994adb,client_7f2253d7e2,7812,stale_visible,REFRESH_REVIEW,194,7812,292,0.01,39.0
5,6,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_visible,REFRESH_REVIEW,193,7558,803,0.20,17.9
6,7,content_b16bd7307b39,client_7f2253d7e2,4590,stale_visible,REFRESH_REVIEW,194,4590,554,0.00,31.0
7,8,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_visible,REFRESH_REVIEW,194,4556,746,0.33,16.4
8,9,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_visible,REFRESH_REVIEW,194,4429,548,0.38,25.3
9,10,content_928af3e22c80,client_7f2253d7e2,1697,stale_visible,REFRESH_REVIEW,193,1697,381,0.12,15.8


### Top-20 review

The top 20 are reviewed as decision-support recommendations rather than automatically correct actions. Each row is selected because its score is high under the stale-and-visible rule.

Confidence reflects how directly the observed signals support the rule. A pick could be wrong if the page is intentionally old, already scheduled for an update, has temporary traffic, or if impressions do not represent a meaningful opportunity for refresh.

In [8]:
# ---------------------------------------------------------
# TOP-20 REVIEW
# ---------------------------------------------------------

top20 = queue.head(20).copy()

reviews = []

for _, row in top20.iterrows():
    action = row["action"]
    reason = row["reason_code"]

    if row["days_since_last_update"] >= 365:
        confidence = "High — strongly stale and still visible."
    else:
        confidence = "Medium — stale and visible, but less extreme."

    what_would_make_wrong = (
        "The recommendation could be wrong if the content is intentionally "
        "left unchanged, already has a planned refresh, or the observed "
        "impressions do not represent a meaningful refresh opportunity."
    )

    reviews.append({
        "rank": int(row["rank"]),
        "content_id": row["content_id"],
        "action": action,
        "reason_code": reason,
        "confidence_note": confidence,
        "what_would_make_it_wrong": what_would_make_wrong
    })

top20_review = pd.DataFrame(reviews)

display(top20_review)

,rank,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...
1,2,content_7368877ea310,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...
2,3,content_1bfaa38ff26c,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...
3,4,content_0a91db491d14,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...
4,5,content_5feee3994adb,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...
5,6,content_c2d929d83eaa,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...
6,7,content_b16bd7307b39,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...
7,8,content_fe16a55cd13d,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...
8,9,content_ecb6215e79fd,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...
9,10,content_928af3e22c80,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...


## 4. Weak picks + leakage check

### Weak picks + leakage check

The main weak picks are items that rank highly because they have large historical visibility, but whose traffic may not represent a genuine refresh opportunity. The rule is intentionally simple, so high impressions alone can dominate the ranking once an item crosses the staleness threshold.

The leakage check confirms that the baseline does not use the declining label or its source fields. Client and content IDs are retained only for identification and grouping, not as predictive features.

In [9]:
# ---------------------------------------------------------
# WEAK PICKS + LEAKAGE CHECK
# ---------------------------------------------------------

print("Potential weak picks from the top 20:")

weak_picks = top20_review[
    top20_review["confidence_note"].str.startswith("Medium")
].copy()

if len(weak_picks) == 0:
    print(
        "No medium-confidence items appeared in the top 20. "
        "Review the most extreme rows manually because a simple rule "
        "can still produce false positives."
    )
else:
    display(weak_picks)

# ---------------------------------------------------------
# LEAKAGE CHECK
# ---------------------------------------------------------

forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

used_features = [
    "days_since_last_update",
    "impressions_90d"
]

print("\nLeakage check:")

for feature in forbidden_features:
    if feature in used_features:
        print("FAIL:", feature)
    else:
        print("PASS — not used:", feature)

print("\nIDs are identifiers only:")
print("PASS — content_id is not used in score calculation.")
print("PASS — client_id is not used in score calculation.")

print("\nFuture-window check:")
print("PASS — baseline uses the supplied snapshot fields only.")

Potential weak picks from the top 20:


,rank,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...
1,2,content_7368877ea310,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...
2,3,content_1bfaa38ff26c,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...
3,4,content_0a91db491d14,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...
4,5,content_5feee3994adb,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...
5,6,content_c2d929d83eaa,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...
6,7,content_b16bd7307b39,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...
7,8,content_fe16a55cd13d,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...
8,9,content_ecb6215e79fd,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...
9,10,content_928af3e22c80,REFRESH_REVIEW,stale_visible,"Medium — stale and visible, but less extreme.",The recommendation could be wrong if the conte...



Leakage check:
PASS — not used: trend_direction
PASS — not used: trend_pct
PASS — not used: is_declining_label

IDs are identifiers only:
PASS — content_id is not used in score calculation.
PASS — client_id is not used in score calculation.

Future-window check:
PASS — baseline uses the supplied snapshot fields only.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.